# Détection de passages piétons — SAM3

Segmentation zero-shot de passages piétons (zebra crossing)  
sur orthophoto IGN 2024 — 20 cm/pixel, tuiles 512×512, EPSG:2154.  
Données : GPSO (Grand Paris Seine Ouest).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mandresyandri/geo-sam3-lab/blob/main/use_cases/pedestrian_crossing/notebook.ipynb)

## 0. Environnement

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    print("Environnement : Google Colab")
except ImportError:
    IN_COLAB = False
    print("Environnement : local (Jupyter)")

In [ ]:
if IN_COLAB:
    import subprocess

    print("Installation geo-sam3-inference...")
    r = subprocess.run(
        ["pip", "install", "-q",
         "git+https://github.com/mandresyandri/geo-sam3-lab.git"],
        capture_output=True, text=True,
    )
    print("OK" if r.returncode == 0 else r.stderr)

    print("Installation ipywidgets...")
    subprocess.run(["pip", "install", "-q", "ipywidgets"], capture_output=True)
    print("OK")
else:
    print("Local — assure-toi d'avoir fait : pip install -e '.[dev,demo]'")

## 1. Config

In [ ]:
MODEL_ID   = "facebook/sam3"
PROMPT     = "pedestrian crossing"
THRESHOLD  = 0.35
OPACITY    = 0.45
COLOR      = (255, 50, 50)
OUTPUT_DIR = "output/"
CACHE_DIR  = "demo_img/"

# INPUT_PATH est défini après téléchargement de la tuile (section 2)
INPUT_PATH = None

## 2. Données — Orthophoto IGN (HuggingFace)

Tuiles GeoTIFF 512×512 stockées sur HuggingFace dans `mandresyandri/orthophoto_2024`.  
Organised par ville (GPSO) : `ortho_tiles/{VILLE}/{tuile}.tif`.

> **HF_TOKEN** : requis si le dépôt est privé.  
> — Colab : ajoute le secret via *Secrets* (clé `HF_TOKEN`).  
> — Local : définis `HF_TOKEN=hf_xxx` dans un fichier `.env` à la racine du projet.

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download, login

HF_REPO_ID   = "mandresyandri/orthophoto_2024"
HF_REPO_TYPE = "dataset"

if IN_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("HuggingFace : connecte")
else:
    print("HuggingFace : acces public (sans HF_TOKEN)")

In [ ]:
api = HfApi()

print(f"Chargement de l'index {HF_REPO_ID}...")
all_files = list(api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE))

cities = sorted(set(
    fname.split("/")[1]
    for fname in all_files
    if fname.startswith("ortho_tiles/") and len(fname.split("/")) >= 3
))

tiles_by_city = {
    city: sorted(
        fname.split("/")[-1]
        for fname in all_files
        if fname.startswith(f"ortho_tiles/{city}/") and fname.lower().endswith(".tif")
    )
    for city in cities
}

print(f"{len(cities)} ville(s) disponible(s) :")
for city in cities:
    print(f"  {city:<35} {len(tiles_by_city[city])} tuile(s)")

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

_style = {"description_width": "60px"}
_layout = widgets.Layout(width="420px")

_default_city  = cities[0] if cities else ""
_default_tiles = tiles_by_city.get(_default_city, [])

city_dd  = widgets.Dropdown(
    options=cities, value=_default_city,
    description="Ville :", style=_style, layout=_layout,
)
tile_dd  = widgets.Dropdown(
    options=_default_tiles,
    description="Tuile :", style=_style, layout=_layout,
)
btn_dl   = widgets.Button(
    description="Telecharger", button_style="primary", icon="download",
    layout=widgets.Layout(width="160px"),
)
out_info = widgets.Output()


def _update_tiles(change):
    tile_dd.options = tiles_by_city.get(change["new"], [])


def _on_download(b):
    global INPUT_PATH
    with out_info:
        clear_output(wait=True)
        city = city_dd.value
        tile = tile_dd.value
        if not tile:
            print("Selectionne une tuile.")
            return
        btn_dl.disabled = True
        btn_dl.description = "En cours..."
        try:
            print(f"Telechargement : ortho_tiles/{city}/{tile}")
            INPUT_PATH = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=f"ortho_tiles/{city}/{tile}",
                repo_type=HF_REPO_TYPE,
                local_dir=CACHE_DIR,
            )
            print(f"OK -> {INPUT_PATH}")
        except Exception as e:
            print(f"Erreur : {e}")
        finally:
            btn_dl.disabled = False
            btn_dl.description = "Telecharger"


city_dd.observe(_update_tiles, names="value")
btn_dl.on_click(_on_download)

display(widgets.VBox([city_dd, tile_dd, btn_dl, out_info]))

In [ ]:
assert INPUT_PATH is not None, "Telecharge d'abord une tuile via le widget ci-dessus."
print(f"Tuile active : {INPUT_PATH}")

## 3. Validation et lecture

In [ ]:
from geo_sam3_inference import GeoImageReader, validate_geotiff

validate_geotiff(INPUT_PATH)
image, geo_meta = GeoImageReader.read(INPUT_PATH)

print(f"CRS       : {geo_meta.crs}")
print(f"Taille    : {geo_meta.width}x{geo_meta.height} px")
print(f"Transform : {geo_meta.transform}")
image

## 4. Inference SAM3

In [ ]:
from geo_sam3_inference import Sam3InferenceEngine

engine = Sam3InferenceEngine(MODEL_ID)
masks  = engine.predict_masks(image, PROMPT, THRESHOLD)

print(f"{len(masks)} masque(s) detecte(s)")

## 5. Visualisation

In [ ]:
from geo_sam3_inference import compute_stats, draw_contours, draw_overlay

viz = draw_overlay(image, masks, COLOR, OPACITY)
viz = draw_contours(viz, masks, COLOR)

stats = compute_stats(masks, image)
print(f"Passages detectes : {stats['count']}")
print(f"Couverture        : {stats['coverage_pct']}%")
for i, area in enumerate(stats['areas']):
    print(f"  Passage #{i+1} : {area:,} px")

viz

## 6. Export geospatial

In [ ]:
from pathlib import Path
from geo_sam3_inference import export_geojson, export_geotiff

out = Path(OUTPUT_DIR)
export_geotiff(masks, geo_meta, out / "pedestrian_crossings.tif")
export_geojson(masks, geo_meta, out / "pedestrian_crossings.geojson")
viz.save(out / "visualization.png")

print("Fichiers exportes :")
for f in sorted(out.iterdir()):
    print(f"  {f.name}")